# Drought-forecasting runbook - Kaggle (Internet ON, WS2M, sharded multi-cell)

End-to-end and aligned with the manuscript Methods (REV3). The multi-cell step is **sharded**: the
leave-one-cell-out transfer is run as one short job per held-out cell (identical results, resumable).

## Before running (right-hand Settings panel)
1. **Internet: ON** (only needed if you let Step 1 download; you can skip it by attaching your data).
2. **Accelerator: GPU T4 x2** (or P100). Without a GPU -> RF + baselines only.
3. **Persistence: Files only** (lets /kaggle/working survive between interactive sessions).
4. **Add Data**: attach your **scripts** Dataset (the 7 .py). Optionally also attach a **data** Dataset
   (your downloaded `cells.csv` + 10 `cell_*.csv`) so you do NOT re-download.


## Setup - copy scripts, (optional) resume previous parts, check runtime

In [ ]:
import os, sys, glob, shutil, urllib.request

hits = glob.glob('/kaggle/input/**/tcn_drought_pipeline_v2.py', recursive=True)
assert hits, "Attach your scripts Dataset (the 7 .py) via 'Add Data'."
SCRIPTS_DIR = os.path.dirname(hits[0]); print('scripts:', SCRIPTS_DIR)
for f in glob.glob(os.path.join(SCRIPTS_DIR, '*.py')):
    shutil.copy(f, '/kaggle/working/')
os.chdir('/kaggle/working'); sys.path.insert(0, '/kaggle/working')

# RESUME (optional): if you added a PREVIOUS run's output as input, carry its transfer parts forward
for prev in glob.glob('/kaggle/input/**/results_mc/parts', recursive=True):
    dst = '/kaggle/working/results_mc/parts'; os.makedirs(dst, exist_ok=True)
    for f in glob.glob(prev + '/*.csv'): shutil.copy(f, dst)
    print('resumed parts from', prev)

try:
    import tensorflow as tf
    print('TensorFlow', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))
except Exception as e:
    print('TensorFlow unavailable -> RF + baselines only:', e)
try:
    urllib.request.urlopen('https://power.larc.nasa.gov', timeout=20); print('Internet: OK')
except Exception as e:
    print('Internet OFF (only needed if you download):', e)

## Data - use an attached data Dataset if present, else download (Step 1)

In [ ]:
import pandas as pd, glob
cand = glob.glob('/kaggle/input/**/cells.csv', recursive=True)
DATA_DIR = os.path.dirname(cand[0]) if cand else 'data'
print('DATA_DIR =', DATA_DIR)

In [ ]:
# Download ONLY if no cell files are present yet (needs Internet ON). Writes ./data with WS2M.
if not glob.glob(os.path.join(DATA_DIR, 'cell_*.csv')):
    !python download_nasapower_cells.py
    DATA_DIR = 'data'
print('using DATA_DIR =', DATA_DIR)

In [ ]:
# verify data + WS2M
print(pd.read_csv(os.path.join(DATA_DIR, 'cells.csv')).to_string(index=False))
print('\nWS2M in EVERY cell:',
      all('WS2M' in pd.read_csv(f, nrows=1).columns for f in glob.glob(os.path.join(DATA_DIR, 'cell_*.csv'))))

In [ ]:
RESULTS    = '/kaggle/working/results'
RESULTS_MC = '/kaggle/working/results_mc'
FIGS       = '/kaggle/working/figures'

## Step A - single-cell pipeline on the WS2M base cell (Central Sumbawa)
SMOKE first (minutes), then FULL. Output -> results/.

In [ ]:
# SMOKE
import importlib, tcn_drought_pipeline_v2 as P; importlib.reload(P)
c = dict(P.CONFIG)
c['data_path'] = os.path.join(DATA_DIR, 'cell_sumbawaC.csv')
c['out_dir']   = '/kaggle/working/results_smoke'
c['seeds'] = [0]; c['horizons'] = [1, 14]
P.main(c)

In [ ]:
# FULL (5 seeds x 4 horizons). To lighten: c['seeds']=[0,1,2]; c['horizons']=[1,14,30]
import importlib, tcn_drought_pipeline_v2 as P; importlib.reload(P)
c = dict(P.CONFIG)
c['data_path'] = os.path.join(DATA_DIR, 'cell_sumbawaC.csv')
c['out_dir']   = RESULTS
P.main(c)

## Step B - multi-cell, SHARDED (resumable; identical to running it all at once)

Order: (1) measure one transfer cell, (2) loop the remaining transfer cells, (3) in-domain whole,
(4) merge. Each transfer job trains on the other 9 cells and tests the held-out cell, and saves a
partial CSV under `results_mc/parts/`. If a session stops, re-run the loop - finished cells are skipped.

In [ ]:
# (1) MEASURE one transfer cell to estimate total time on this GPU
import time, importlib, tcn_drought_multicell as M; importlib.reload(M)
t = time.time()
c = dict(M.CONFIG_MC); c['data_dir'] = DATA_DIR; c['out_dir'] = RESULTS_MC
c['phase'] = 'transfer'; c['only_held'] = 'lombok'
# To make each shard lighter (keep these consistent in the loop below):
# c['transfer_models'] = ['RandomForest']; c['seeds'] = [0,1,2]; c['only_horizons'] = [1,14,30]
M.main(c)
print(f'\nONE transfer cell ~ {(time.time()-t)/60:.1f} min  ->  x10 cells + single-cell full + in-domain = your total')

In [ ]:
# (2) LOOP the remaining transfer cells (skips any already saved in results_mc/parts/)
import importlib, tcn_drought_multicell as M; importlib.reload(M)
CELLS = ['lombok','sumbawaB','sumbawaC','bima','manggarai','ende','floresT','sumba','kupang','soe']
done = {os.path.basename(p).split('__')[1] for p in glob.glob(os.path.join(RESULTS_MC, 'parts', 'transfer__*'))}
for held in CELLS:
    if held in done:
        print('skip (already done):', held); continue
    c = dict(M.CONFIG_MC); c['data_dir'] = DATA_DIR; c['out_dir'] = RESULTS_MC
    c['phase'] = 'transfer'; c['only_held'] = held
    # If you lightened the measure cell, set the SAME options here:
    # c['transfer_models'] = ['RandomForest']; c['seeds'] = [0,1,2]; c['only_horizons'] = [1,14,30]
    M.main(c)

In [ ]:
# (3) IN-DOMAIN + regional alert (light; run whole)
import importlib, tcn_drought_multicell as M; importlib.reload(M)
c = dict(M.CONFIG_MC); c['data_dir'] = DATA_DIR; c['out_dir'] = RESULTS_MC; c['phase'] = 'indomain'
M.main(c)

In [ ]:
# (4) MERGE -> metrics_transfer.csv + skill_map.csv (in-domain/regional already written)
import importlib, tcn_drought_multicell as M; importlib.reload(M)
c = dict(M.CONFIG_MC); c['data_dir'] = DATA_DIR; c['out_dir'] = RESULTS_MC; c['phase'] = 'merge'
M.main(c)

## Step C - figures (1-6 + Results 7-10)

In [ ]:
os.environ['BASE_CELL_CSV'] = os.path.join(DATA_DIR, 'cell_sumbawaC.csv')
os.environ['FIG_OUT']       = FIGS
!python make_all_figures_pro.py        # Figures 1-6

import importlib, make_results_figures as R; importlib.reload(R)
R.RESULTS_DIR = RESULTS                 # use '/kaggle/working/results_smoke' if only smoke ran
R.OUT_DIR     = FIGS
os.makedirs(FIGS, exist_ok=True)
R.fig_skill_vs_horizon(); R.fig_pred_vs_obs(); R.fig_dss_performance(); R.fig_forecast_vs_warning()
print('figures ->', sorted(os.listdir(FIGS)))

## Step D - bundle outputs for download (Output panel)

In [ ]:
import shutil
for d in [RESULTS, RESULTS_MC, FIGS]:
    if os.path.isdir(d):
        shutil.make_archive(d, 'zip', d); print('zipped', os.path.basename(d) + '.zip')
print('Download the .zip files from the Kaggle Output panel.')